# 03 — Whole Hospital Flow

Author: Saige Mukherjee

Contact: mukherjeesaige@gmail.com //
https://www.linkedin.com/in/saige-mukherjee-0aba68281/

The analysis runs inside the Jupyter notebook. Run the notebook with credentialed access to generate the results.

If you want to run the notebook and generate the report:
- obtained credentialed access to MIMIC-IV via PhysioNet and BigQuery,
- create a GCP project and star the MIMIC-IV dataset ,
- enter the GCP project ID below and execute the program.

In [ ]:
# Enter a Google Cloud billing project authorized to query MIMIC-IV.
PROJECT_ID = ""


---

## 0. Environment setup

Run in an environment authenticated to Google Cloud.


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from google.cloud import bigquery
from IPython.display import display, Markdown

HOSP = "physionet-data.mimiciv_3_1_hosp"
ICU = "physionet-data.mimiciv_3_1_icu"

INCLUDE_ED_ONLY = True
THRESHOLDS_HOURS = [48, 72, 168, 336]
MAX_SEGMENT_HOURS = 365 * 24
MIN_CELL_N = 11

if not PROJECT_ID.strip():
    raise ValueError("Set PROJECT_ID before running the notebook.")

if INCLUDE_ED_ONLY is not True:
    raise ValueError("This notebook is intended to include ED-only encounters.")

client = bigquery.Client(project=PROJECT_ID)

print(f"Hospital dataset: {HOSP}")
print(f"ICU dataset:      {ICU}")
print(f"Include ED-only rows with NULL hadm_id: {INCLUDE_ED_ONLY}")
print(f"Maximum valid segment duration: {MAX_SEGMENT_HOURS:,} hours")


---

## 1. Helper functions

`safe_display()` suppresses aggregate rows with counts from 1 through 10.


In [ ]:
def run_query(sql: str, job_label: str | None = None) -> pd.DataFrame:
    """Run a BigQuery query and return a pandas DataFrame."""
    job_config = bigquery.QueryJobConfig()
    if job_label:
        job_config.labels = {
            "notebook": "03-whole-hospital-flow",
            "step": job_label[:63].lower().replace("_", "-"),
        }
    return client.query(sql, job_config=job_config).to_dataframe()


def likely_count_columns(df: pd.DataFrame) -> list[str]:
    """Identify columns that likely contain aggregate counts."""
    count_like = []
    for col in df.columns:
        c = col.lower()
        if (
            c in {
                "n", "count", "segments", "unit_stays", "transfer_count",
                "admissions", "hospitalizations", "patients",
                "unique_patients", "unique_admissions",
                "segments_over_threshold",
                "segments_excluded_over_365d"
            }
            or c.startswith("n_")
            or c.endswith("_n")
            or c.endswith("_count")
        ):
            # Check if it is numeric OR if it is an object column that could contain numbers
            if pd.api.types.is_numeric_dtype(df[col]) or df[col].dtype == object:
                count_like.append(col)
    return count_like


def suppress_small_cells(df: pd.DataFrame, min_n: int = MIN_CELL_N) -> pd.DataFrame:
    """Mask aggregate cells where a detected count is between 1 and min_n - 1 with '<11'."""
    out = df.copy().astype(object)
    count_cols = likely_count_columns(df)
    if not count_cols:
        return out

    for col in count_cols:
        # Convert to numeric to ensure comparison works, then mask
        vals = pd.to_numeric(df[col], errors='coerce')
        mask = (vals > 0) & (vals < min_n)
        out.loc[mask, col] = f"<{min_n}"

    return out


def safe_display(df: pd.DataFrame, min_n: int = MIN_CELL_N, max_rows: int = 50):
    """Display an aggregate table after small-cell masking."""
    shown = suppress_small_cells(df, min_n=min_n)

    # Check for masked values in the result
    is_masked = (shown == f"<{min_n}").any(axis=1)
    num_masked_rows = is_masked.sum()

    if num_masked_rows:
        display(Markdown(
            f"**Masked values in {num_masked_rows} row(s) where `0 < n < {min_n}`.**"
        ))

    with pd.option_context(
        "display.max_rows", max_rows,
        "display.max_columns", 50,
        "display.width", 180
    ):
        display(shown.head(max_rows))

---

## 2. Reusable whole-flow transfer cohort

The cohort below reads directly from `hosp.transfers`.

It intentionally contains:

- no join to `hosp.admissions`;
- no `hadm_id IS NOT NULL` condition;
- no exclusion based on admission status.

The condition `AND (TRUE)` is generated because `INCLUDE_ED_ONLY = True`.


In [ ]:
scope_filter_sql = "TRUE" if INCLUDE_ED_ONLY else "hadm_id IS NOT NULL"

TRANSFER_COHORT_CTE = f"""
WITH transfer_candidates AS (
    SELECT
        subject_id,
        hadm_id,
        transfer_id,
        eventtype,
        careunit,
        intime,
        outtime,
        TIMESTAMP_DIFF(outtime, intime, SECOND) / 3600.0 AS hours_in_careunit
    FROM `{HOSP}.transfers`
    WHERE careunit IS NOT NULL
      AND intime IS NOT NULL
      AND outtime IS NOT NULL
      AND outtime > intime
      AND ({scope_filter_sql})
),
transfer_cohort AS (
    SELECT *
    FROM transfer_candidates
    WHERE hours_in_careunit > 0
      AND hours_in_careunit <= {MAX_SEGMENT_HOURS}
)
"""

print(TRANSFER_COHORT_CTE)

assert "hadm_id IS NOT NULL" not in TRANSFER_COHORT_CTE
assert "JOIN" not in TRANSFER_COHORT_CTE.upper()


---

## 3. Cohort audit and ED-inclusion proof

The first table verifies the complete cohort. The second table separately reports ED-only and admission-linked components so the inclusion is visible in the notebook output.


In [ ]:
sql_scope_audit = f"""
WITH transfer_candidates AS (
    SELECT
        subject_id,
        hadm_id,
        careunit,
        intime,
        outtime,
        TIMESTAMP_DIFF(outtime, intime, SECOND) / 3600.0 AS hours_in_careunit
    FROM `{HOSP}.transfers`
    WHERE careunit IS NOT NULL
      AND intime IS NOT NULL
      AND outtime IS NOT NULL
      AND outtime > intime
),
cohort AS (
    SELECT *
    FROM transfer_candidates
    WHERE hours_in_careunit > 0
      AND hours_in_careunit <= {MAX_SEGMENT_HOURS}
)
SELECT
    (SELECT COUNT(*) FROM `{HOSP}.transfers`) AS source_transfer_rows,
    (SELECT COUNT(*) FROM transfer_candidates) AS positive_duration_candidates,
    (SELECT COUNTIF(hadm_id IS NULL) FROM transfer_candidates)
        AS ed_only_candidates,
    (SELECT COUNTIF(hours_in_careunit > {MAX_SEGMENT_HOURS}) FROM transfer_candidates)
        AS segments_excluded_over_365d,
    (SELECT COUNT(*) FROM cohort) AS valid_segments,
    (SELECT COUNTIF(hadm_id IS NULL) FROM cohort) AS ed_only_segments,
    (SELECT COUNTIF(hadm_id IS NOT NULL) FROM cohort) AS admission_linked_segments,
    (SELECT COUNT(DISTINCT subject_id) FROM cohort) AS unique_patients,
    (SELECT COUNT(DISTINCT hadm_id) FROM cohort) AS represented_admissions,
    (SELECT COUNT(DISTINCT careunit) FROM cohort) AS distinct_careunits
"""

scope_audit_df = run_query(sql_scope_audit, "scope_audit")
# Display with masking
safe_display(scope_audit_df)

sql_scope_components = f"""
{TRANSFER_COHORT_CTE}
SELECT
    CASE
        WHEN hadm_id IS NULL THEN 'ED-only: hadm_id is NULL'
        ELSE 'Admission-linked: hadm_id is present'
    END AS cohort_component,
    COUNT(*) AS segments,
    ROUND(100 * SAFE_DIVIDE(COUNT(*), SUM(COUNT(*)) OVER ()), 2)
        AS pct_of_segments,
    ROUND(APPROX_QUANTILES(hours_in_careunit, 100)[OFFSET(50)], 2)
        AS median_hours,
    ROUND(SUM(hours_in_careunit), 2) AS total_segment_hours
FROM transfer_cohort
GROUP BY cohort_component
ORDER BY cohort_component
"""

scope_components_df = run_query(sql_scope_components, "scope_components")
safe_display(scope_components_df)

In [ ]:
expected_segments = 1_867_366
expected_ed_only = 408_882
expected_admission_linked = 1_458_484
expected_admissions = 545_994
expected_over_365_str = '<11'

row = scope_audit_df.iloc[0]

def check_val(obs, exp_str):
    if exp_str == '<11':
        return (obs > 0) and (obs < 11)
    return int(obs) == int(exp_str)

# Force the masked version for the validation display
audit_masked = suppress_small_cells(scope_audit_df)

checks = pd.DataFrame({
    "check": [
        "All valid segments",
        "ED-only segments included",
        "Admission-linked segments",
        "Represented admissions",
        "Segments excluded over 365 days",
    ],
    "observed": [
        int(row["valid_segments"]),
        int(row["ed_only_segments"]),
        int(row["admission_linked_segments"]),
        int(row["represented_admissions"]),
        audit_masked.iloc[0]["segments_excluded_over_365d"], # Shows <11
    ],
    "matches_expected": [
        int(row["valid_segments"]) == expected_segments,
        int(row["ed_only_segments"]) == expected_ed_only,
        int(row["admission_linked_segments"]) == expected_admission_linked,
        int(row["represented_admissions"]) == expected_admissions,
        check_val(row["segments_excluded_over_365d"], expected_over_365_str),
    ]
})

display(checks)

assert int(row["ed_only_segments"]) > 0
assert int(row["valid_segments"]) == (int(row["ed_only_segments"]) + int(row["admission_linked_segments"]))

if not checks["matches_expected"].all():
    print("Prior validation targets missed.")
else:
    print("ED-only rows included and small counts masked.")

---

## 4. Whole-hospital duration summary

These statistics describe care-location segments, not full admission length of stay.

Because the cohort includes ED-only encounters, `admissions` counts only the subset of segments that have a non-null `hadm_id`.


In [ ]:
sql_duration_summary = f"""
{TRANSFER_COHORT_CTE}
SELECT
    COUNT(*) AS segments,
    COUNTIF(hadm_id IS NULL) AS ed_only_segments,
    COUNTIF(hadm_id IS NOT NULL) AS admission_linked_segments,
    COUNT(DISTINCT hadm_id) AS represented_admissions,
    COUNT(DISTINCT subject_id) AS patients,
    ROUND(SUM(hours_in_careunit), 2) AS total_segment_hours,
    ROUND(AVG(hours_in_careunit), 2) AS mean_hours,
    ROUND(APPROX_QUANTILES(hours_in_careunit, 100)[OFFSET(50)], 2) AS median_hours,
    ROUND(APPROX_QUANTILES(hours_in_careunit, 100)[OFFSET(75)], 2) AS p75_hours,
    ROUND(APPROX_QUANTILES(hours_in_careunit, 100)[OFFSET(90)], 2) AS p90_hours,
    ROUND(APPROX_QUANTILES(hours_in_careunit, 100)[OFFSET(95)], 2) AS p95_hours,
    ROUND(APPROX_QUANTILES(hours_in_careunit, 100)[OFFSET(99)], 2) AS p99_hours,
    ROUND(MAX(hours_in_careunit), 2) AS max_hours
FROM transfer_cohort
"""

duration_summary_df = run_query(sql_duration_summary, "duration_summary")
safe_display(duration_summary_df)


The whole-flow cohort should produce a median segment near **8.8 hours**. This is lower than the approximately **16.6-hour** median for the admitted-hospitalization-only cohort because ED-only encounters are included here.


---

## 5. Duration distribution by operational bin

The binning occurs in BigQuery, so no patient-level duration rows are downloaded.


In [ ]:
sql_duration_bins = f"""
{TRANSFER_COHORT_CTE},
binned AS (
    SELECT
        CASE
            WHEN hours_in_careunit < 1 THEN '<1h'
            WHEN hours_in_careunit < 4 THEN '1-4h'
            WHEN hours_in_careunit < 8 THEN '4-8h'
            WHEN hours_in_careunit < 12 THEN '8-12h'
            WHEN hours_in_careunit < 24 THEN '12-24h'
            WHEN hours_in_careunit < 48 THEN '24-48h'
            WHEN hours_in_careunit < 72 THEN '48-72h'
            WHEN hours_in_careunit < 168 THEN '72h-7d'
            WHEN hours_in_careunit < 336 THEN '7-14d'
            ELSE '14d-365d'
        END AS duration_bin,
        CASE
            WHEN hours_in_careunit < 1 THEN 1
            WHEN hours_in_careunit < 4 THEN 2
            WHEN hours_in_careunit < 8 THEN 3
            WHEN hours_in_careunit < 12 THEN 4
            WHEN hours_in_careunit < 24 THEN 5
            WHEN hours_in_careunit < 48 THEN 6
            WHEN hours_in_careunit < 72 THEN 7
            WHEN hours_in_careunit < 168 THEN 8
            WHEN hours_in_careunit < 336 THEN 9
            ELSE 10
        END AS bin_order,
        hours_in_careunit
    FROM transfer_cohort
)
SELECT
    duration_bin,
    bin_order,
    COUNT(*) AS segments,
    ROUND(100 * SAFE_DIVIDE(COUNT(*), SUM(COUNT(*)) OVER ()), 2)
        AS pct_of_segments,
    ROUND(SUM(hours_in_careunit), 2) AS segment_hours,
    ROUND(
        100 * SAFE_DIVIDE(
            SUM(hours_in_careunit),
            SUM(SUM(hours_in_careunit)) OVER ()
        ),
        2
    ) AS pct_of_segment_hours
FROM binned
GROUP BY duration_bin, bin_order
ORDER BY bin_order
"""

duration_bins_df = run_query(sql_duration_bins, "duration_bins")
safe_display(duration_bins_df)


In [ ]:
plot_df = suppress_small_cells(duration_bins_df)

plt.figure(figsize=(10, 5))
plt.bar(plot_df["duration_bin"], plot_df["pct_of_segments"])
plt.ylabel("Share of valid segments (%)")
plt.xlabel("Segment duration")
plt.title("Distribution of whole-flow care-unit segments")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.show()


---

## 6. Threshold burden

For each threshold, this reports both the share of segments and the share of all segment-hours contained in those long segments. It also calculates excess time beyond the threshold.


In [ ]:
threshold_list_sql = ", ".join(str(x) for x in THRESHOLDS_HOURS)

sql_thresholds = f"""
{TRANSFER_COHORT_CTE},
thresholds AS (
    SELECT threshold_hours
    FROM UNNEST([{threshold_list_sql}]) AS threshold_hours
)
SELECT
    threshold_hours,
    COUNTIF(c.hours_in_careunit > threshold_hours) AS segments_over_threshold,
    ROUND(
        100 * SAFE_DIVIDE(
            COUNTIF(c.hours_in_careunit > threshold_hours),
            COUNT(*)
        ),
        2
    ) AS pct_of_segments,
    ROUND(
        SUM(IF(c.hours_in_careunit > threshold_hours, c.hours_in_careunit, 0)),
        2
    ) AS hours_in_segments_over_threshold,
    ROUND(
        100 * SAFE_DIVIDE(
            SUM(IF(c.hours_in_careunit > threshold_hours, c.hours_in_careunit, 0)),
            SUM(c.hours_in_careunit)
        ),
        2
    ) AS pct_of_segment_hours,
    ROUND(
        SUM(GREATEST(c.hours_in_careunit - threshold_hours, 0)),
        2
    ) AS excess_hours_beyond_threshold
FROM transfer_cohort AS c
CROSS JOIN thresholds
GROUP BY threshold_hours
ORDER BY threshold_hours
"""

threshold_df = run_query(sql_thresholds, "threshold_burden")
safe_display(threshold_df)


In [ ]:
plot_df = threshold_df.copy()
labels = [
    f"{int(h)}h" if h < 168 else f"{int(h / 24)}d"
    for h in plot_df["threshold_hours"]
]
x = np.arange(len(labels))
width = 0.38

plt.figure(figsize=(9, 5))
plt.bar(x - width / 2, plot_df["pct_of_segments"], width, label="Share of segments")
plt.bar(x + width / 2, plot_df["pct_of_segment_hours"], width, label="Share of segment-hours")
plt.xticks(x, labels)
plt.ylabel("Percent")
plt.xlabel("Duration threshold")
plt.title("Long segments consume a disproportionate share of recorded capacity")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(9, 5))
plt.bar(labels, threshold_df["excess_hours_beyond_threshold"] / 1_000_000)
plt.ylabel("Excess hours (millions)")
plt.xlabel("Duration threshold")
plt.title("Whole-hospital excess care-unit hours by threshold")
plt.tight_layout()
plt.show()


---

## 7. Event type summary

`eventtype` is descriptive metadata and should not be interpreted as a complete explanation for the clinical or operational meaning of a segment.


In [ ]:
sql_eventtypes = f"""
{TRANSFER_COHORT_CTE}
SELECT
    COALESCE(eventtype, 'Missing') AS eventtype,
    COUNT(*) AS segments,
    ROUND(100 * SAFE_DIVIDE(COUNT(*), SUM(COUNT(*)) OVER ()), 2)
        AS pct_of_segments,
    ROUND(APPROX_QUANTILES(hours_in_careunit, 100)[OFFSET(50)], 2)
        AS median_hours,
    ROUND(APPROX_QUANTILES(hours_in_careunit, 100)[OFFSET(75)], 2)
        AS p75_hours,
    ROUND(APPROX_QUANTILES(hours_in_careunit, 100)[OFFSET(90)], 2)
        AS p90_hours,
    ROUND(APPROX_QUANTILES(hours_in_careunit, 100)[OFFSET(95)], 2)
        AS p95_hours,
    ROUND(100 * AVG(IF(hours_in_careunit > 48, 1, 0)), 2)
        AS pct_over_48h,
    ROUND(100 * AVG(IF(hours_in_careunit > 72, 1, 0)), 2)
        AS pct_over_72h
FROM transfer_cohort
GROUP BY eventtype
ORDER BY segments DESC
"""

eventtypes_df = run_query(sql_eventtypes, "eventtypes")
safe_display(eventtypes_df)


---

## 8. Raw care-unit burden

This ranks care units by excess hours above 72 hours. It identifies where the long-tail burden is located; it does not prove that the excess hours were avoidable.


In [ ]:
sql_careunits = f"""
{TRANSFER_COHORT_CTE}
SELECT
    careunit,
    COUNT(*) AS segments,
    ROUND(SUM(hours_in_careunit), 2) AS total_segment_hours,
    ROUND(AVG(hours_in_careunit), 2) AS mean_hours,
    ROUND(APPROX_QUANTILES(hours_in_careunit, 100)[OFFSET(50)], 2)
        AS median_hours,
    ROUND(APPROX_QUANTILES(hours_in_careunit, 100)[OFFSET(75)], 2)
        AS p75_hours,
    ROUND(APPROX_QUANTILES(hours_in_careunit, 100)[OFFSET(90)], 2)
        AS p90_hours,
    ROUND(APPROX_QUANTILES(hours_in_careunit, 100)[OFFSET(95)], 2)
        AS p95_hours,
    ROUND(100 * AVG(IF(hours_in_careunit > 72, 1, 0)), 2)
        AS pct_over_72h,
    ROUND(SUM(GREATEST(hours_in_careunit - 72, 0)), 2)
        AS excess_hours_over_72h
FROM transfer_cohort
GROUP BY careunit
HAVING COUNT(*) >= {MIN_CELL_N}
ORDER BY excess_hours_over_72h DESC
"""

careunit_df = run_query(sql_careunits, "careunit_burden")
safe_display(careunit_df, max_rows=50)


In [ ]:
top = careunit_df.head(15).sort_values("excess_hours_over_72h")

plt.figure(figsize=(10, 7))
plt.barh(top["careunit"], top["excess_hours_over_72h"] / 1_000_000)
plt.xlabel("Excess hours beyond 72h (millions)")
plt.ylabel("Care unit")
plt.title("Top care units by excess care-unit hours above 72h")
plt.tight_layout()
plt.show()


---

## 9. Broad care-unit grouping

The mapping is intentionally transparent. Unmapped labels remain visible for QA.


In [ ]:
CAREUNIT_GROUP_CASE = r"""
CASE
    WHEN careunit IN (
        'Emergency Department',
        'Emergency Department Observation'
    ) THEN 'Emergency / ED Observation'

    WHEN careunit IN (
        'Medical Intensive Care Unit (MICU)',
        'Medical/Surgical Intensive Care Unit (MICU/SICU)',
        'Surgical Intensive Care Unit (SICU)',
        'Cardiac Vascular Intensive Care Unit (CVICU)',
        'Coronary Care Unit (CCU)',
        'Neuro Surgical Intensive Care Unit (Neuro SICU)',
        'Trauma SICU (TSICU)'
    ) THEN 'ICU / Critical Care'

    WHEN careunit IN (
        'Medicine',
        'Medicine/Cardiology',
        'Medicine/Cardiology Intermediate',
        'Hematology/Oncology',
        'Oncology',
        'Medical/Surgical (Gynecology)',
        'Transplant',
        'Vascular',
        'Neurology'
    ) THEN 'Medical Ward / Specialty Ward'

    WHEN careunit IN (
        'Surgery',
        'Surgery/Trauma',
        'Med/Surg/Trauma',
        'Orthopaedics',
        'Thoracic Surgery',
        'Cardiac Surgery',
        'PACU'
    ) THEN 'Surgical / Trauma'

    WHEN careunit IN (
        'Psychiatry',
        'Psychiatry Ward'
    ) THEN 'Psychiatry'

    WHEN careunit IN (
        'Obstetrics (Postpartum & Antepartum)',
        'Labor & Delivery'
    ) THEN 'Obstetrics'

    WHEN careunit IN (
        'Discharge Lounge',
        'Observation'
    ) THEN 'Discharge / Throughput'

    ELSE 'Other / Unmapped'
END
"""
print(CAREUNIT_GROUP_CASE)


In [ ]:
sql_group_burden = f"""
{TRANSFER_COHORT_CTE},
grouped AS (
    SELECT
        {CAREUNIT_GROUP_CASE} AS broad_careunit_group,
        hours_in_careunit
    FROM transfer_cohort
)
SELECT
    broad_careunit_group,
    COUNT(*) AS segments,
    ROUND(SUM(hours_in_careunit), 2) AS total_segment_hours,
    ROUND(AVG(hours_in_careunit), 2) AS mean_hours,
    ROUND(APPROX_QUANTILES(hours_in_careunit, 100)[OFFSET(50)], 2)
        AS median_hours,
    ROUND(APPROX_QUANTILES(hours_in_careunit, 100)[OFFSET(75)], 2)
        AS p75_hours,
    ROUND(APPROX_QUANTILES(hours_in_careunit, 100)[OFFSET(90)], 2)
        AS p90_hours,
    ROUND(APPROX_QUANTILES(hours_in_careunit, 100)[OFFSET(95)], 2)
        AS p95_hours,
    ROUND(100 * AVG(IF(hours_in_careunit > 48, 1, 0)), 2)
        AS pct_over_48h,
    ROUND(100 * AVG(IF(hours_in_careunit > 72, 1, 0)), 2)
        AS pct_over_72h,
    ROUND(SUM(GREATEST(hours_in_careunit - 72, 0)), 2)
        AS excess_hours_over_72h
FROM grouped
GROUP BY broad_careunit_group
HAVING COUNT(*) >= {MIN_CELL_N}
ORDER BY excess_hours_over_72h DESC
"""

group_burden_df = run_query(sql_group_burden, "group_burden")
safe_display(group_burden_df)


In [ ]:
plot_df = group_burden_df.sort_values("excess_hours_over_72h")

plt.figure(figsize=(10, 6))
plt.barh(
    plot_df["broad_careunit_group"],
    plot_df["excess_hours_over_72h"] / 1_000_000,
)
plt.xlabel("Excess hours beyond 72h (millions)")
plt.ylabel("Broad care-unit group")
plt.title("Broad groups by excess care-unit hours above 72h")
plt.tight_layout()
plt.show()


---

## 10. Unmapped care-unit review

Update the mapping above when `Other / Unmapped` contains a material share of segments or burden.


In [ ]:
sql_unmapped = f"""
{TRANSFER_COHORT_CTE},
mapped AS (
    SELECT
        careunit,
        {CAREUNIT_GROUP_CASE} AS broad_careunit_group,
        hours_in_careunit
    FROM transfer_cohort
)
SELECT
    careunit,
    COUNT(*) AS segments,
    ROUND(APPROX_QUANTILES(hours_in_careunit, 100)[OFFSET(50)], 2)
        AS median_hours,
    ROUND(SUM(GREATEST(hours_in_careunit - 72, 0)), 2)
        AS excess_hours_over_72h
FROM mapped
WHERE broad_careunit_group = 'Other / Unmapped'
GROUP BY careunit
HAVING COUNT(*) >= {MIN_CELL_N}
ORDER BY excess_hours_over_72h DESC
"""

unmapped_df = run_query(sql_unmapped, "unmapped_review")
safe_display(unmapped_df, max_rows=100)


---

## 11. Approximate-period trend

Because patient dates are independently shifted, raw calendar years are not comparable across patients. This section uses `patients.anchor_year_group`.

The join is by `subject_id`, so ED-only segments remain represented even though they do not have a hospital admission ID.


In [ ]:
sql_period = f"""
{TRANSFER_COHORT_CTE}
SELECT
    p.anchor_year_group,
    COUNT(*) AS segments,
    COUNT(DISTINCT c.hadm_id) AS admissions,
    ROUND(APPROX_QUANTILES(c.hours_in_careunit, 100)[OFFSET(50)], 2)
        AS median_hours,
    ROUND(100 * AVG(IF(c.hours_in_careunit > 72, 1, 0)), 2)
        AS pct_over_72h,
    ROUND(SUM(GREATEST(c.hours_in_careunit - 72, 0)), 2)
        AS excess_hours_over_72h
FROM transfer_cohort AS c
INNER JOIN `{HOSP}.patients` AS p
    ON c.subject_id = p.subject_id
GROUP BY p.anchor_year_group
HAVING COUNT(*) >= {MIN_CELL_N}
ORDER BY p.anchor_year_group
"""

period_df = run_query(sql_period, "anchor_year_group")
safe_display(period_df)


---

## 12. Internal consistency checks

These checks verify that:

- the cohort contains the expected ED-only component;
- every segment has a care unit;
- no duration is non-positive;
- no duration exceeds the 365-day cap;
- the share of segment-hours over a threshold is not smaller than the share of segments over that threshold.


In [ ]:
sql_validation = f"""
{TRANSFER_COHORT_CTE}
SELECT
    COUNTIF(hadm_id IS NULL) AS ed_only_segments,
    COUNTIF(hadm_id IS NOT NULL) AS admission_linked_segments,
    COUNTIF(careunit IS NULL) AS null_careunit_rows,
    COUNTIF(hours_in_careunit <= 0) AS nonpositive_duration_rows,
    COUNTIF(hours_in_careunit > {MAX_SEGMENT_HOURS}) AS over_cap_rows,
    COUNT(*) AS segments
FROM transfer_cohort
"""

validation_df = run_query(sql_validation, "validation")
safe_display(validation_df)

assert INCLUDE_ED_ONLY is True
assert int(validation_df.loc[0, "ed_only_segments"]) == expected_ed_only
assert int(validation_df.loc[0, "admission_linked_segments"]) == expected_admission_linked
assert int(validation_df.loc[0, "segments"]) == expected_segments
assert int(validation_df.loc[0, "null_careunit_rows"]) == 0
assert int(validation_df.loc[0, "nonpositive_duration_rows"]) == 0
assert int(validation_df.loc[0, "over_cap_rows"]) == 0
assert (
    int(validation_df.loc[0, "ed_only_segments"])
    + int(validation_df.loc[0, "admission_linked_segments"])
    == int(validation_df.loc[0, "segments"])
)
assert (threshold_df["pct_of_segment_hours"] >= threshold_df["pct_of_segments"]).all()

print("Validation passed: ED-only rows are included.")


---

## 13. Interpretation

The whole-flow cohort supports the following hospital-operations interpretation:

1. The typical transfer-table care-location segment is much shorter than the long right tail.
2. ED-only encounters are part of the hospital's broader patient-flow footprint and materially affect the overall distribution.
3. A minority of long segments consumes a disproportionate share of recorded care-unit time.
4. Excess time above an operational threshold is concentrated in a subset of care units.
5. These findings locate burden but do not determine when a patient became medically ready or which hours were avoidable.

For inpatient-only or discharge-destination analyses, use a separate cohort restricted to non-null `hadm_id`. To identify true discharge delay, additional structured timestamps would be required for medical readiness, referral, authorization, acceptance, downstream bed availability, transport booking, and actual departure.


---

## Publication safeguard

Before publishing a rendered notebook:

- confirm that every displayed table is aggregate-only;
- suppress groups with `0 < n < 11`;
- do not export patient-, admission-, transfer-, or time-bin-level datasets;
- remove credentials and keep `PROJECT_ID = ""`;
- clear any accidental raw query previews.


## v1.1 - Cohort definition

The analytical unit is a valid row from `hosp.transfers` that:

- has a non-null `careunit`, `intime`, and `outtime`;
- has `outtime > intime`;
- lasts no longer than 365 days.

### ED inclusion

Rows with `hadm_id IS NULL` are **retained**. In MIMIC-IV, these represent ED-only encounters that did not proceed to a hospital admission.

The cohort therefore includes both:

1. ED-only encounters with `hadm_id IS NULL`;
2. transfer segments linked to admitted hospitalizations.

There is no join to `admissions` and no `hadm_id IS NOT NULL` filter in the cohort CTE.